# Token Sequence Workbench

Run tokens one by one, inspect `State` after each step, evaluate forecasts, and prototype new tokens.

---
## Part 1 — Environment Setup

In [ ]:
import os, sys

%cd /content

if not os.path.exists('graph_Time_series'):
    !git clone https://github.com/chahineNejm/graph_Time_series
if not os.path.exists('kernels_playground'):
    !git clone https://github.com/chahineNejm/kernels_playground

for p in ['/content/graph_Time_series',
          '/content/kernels_playground',
          '/content/kernels_playground/first_tests']:
    if p not in sys.path:
        sys.path.insert(0, p)

!pip install -q datasets properscoring

print('Ready.')

---
## Part 2 — Load Token Framework

In [ ]:
from pathlib import Path
import importlib.util
import types
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

PACKAGE_DIR = Path('/content/graph_Time_series/graph_Time_series')


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


pkg = types.ModuleType('graph_Time_series')
pkg.__path__ = [str(PACKAGE_DIR)]
sys.modules.setdefault('graph_Time_series', pkg)

blocks = types.ModuleType('graph_Time_series.token_blocks')
blocks.__path__ = [str(PACKAGE_DIR / 'token_blocks')]
sys.modules.setdefault('graph_Time_series.token_blocks', blocks)

state_mod = load_module('graph_Time_series.state', PACKAGE_DIR / 'state.py')
token_mod = load_module('graph_Time_series.token', PACKAGE_DIR / 'token.py')
norm_mod  = load_module('graph_Time_series.token_blocks.normalization',
                        PACKAGE_DIR / 'token_blocks' / 'normalization.py')
rbf_mod   = load_module('graph_Time_series.token_blocks.kernel_rbf',
                        PACKAGE_DIR / 'token_blocks' / 'kernel_rbf.py')

State                = state_mod.State
TransformRecord      = state_mod.TransformRecord
ZNormalizationToken  = norm_mod.ZNormalizationToken
KernelRBFToken       = rbf_mod.KernelRBFToken

Token          = token_mod.Token
TransformToken = token_mod.TransformToken
FeatureToken   = token_mod.FeatureToken
ModelToken     = token_mod.ModelToken

print('Loaded from', PACKAGE_DIR)

---
## Part 3 — Load Data

Uses `kernels_playground` utilities (same pattern as `tweaks.ipynb`).

In [ ]:
from utils.config import DATASETS
from utils.data import build_examples

DATA_CONFIG = 'electricity_H_long'
N_EXAMPLES_TO_DOWNLOAD = 48
MAX_SAMPLES = 32
HOLDOUT     = 8
HISTORY_LENGTH = 240   # keep only the last N history points
FUTURE_LENGTH  = None  # None keeps the shortest available future length

raw = build_examples(
    config=DATA_CONFIG,
    start=0, stop=N_EXAMPLES_TO_DOWNLOAD, step=1,
    dataset_name=DATASETS['eval'],
)
print(f'Loaded {len(raw)} raw examples')

examples = raw[:MAX_SAMPLES]
min_hist = min(len(e['history']) for e in examples)
min_fut  = min(len(e['future'])  for e in examples)
history_len = min(HISTORY_LENGTH, min_hist) if HISTORY_LENGTH is not None else min_hist
future_len  = min(FUTURE_LENGTH, min_fut) if FUTURE_LENGTH is not None else min_fut

H_all = np.stack([e['history'][-history_len:] for e in examples]).astype(np.float32)
F_all = np.stack([e['future'][:future_len]   for e in examples]).astype(np.float32)

H, F                     = H_all[:-HOLDOUT], F_all[:-HOLDOUT]
H_holdout, F_holdout     = H_all[-HOLDOUT:], F_all[-HOLDOUT:]

print(f'Data config:   {DATA_CONFIG}')
print(f'Downloaded:    {len(raw)} examples')
print(f'Kept lengths:  history={history_len}  future={future_len}')
print(f'Run data:      H {H.shape}  F {F.shape}')
print(f'Held-out data: H {H_holdout.shape}  F {F_holdout.shape}')

---
## Part 4 — State Inspector

`inspect(state)` — full snapshot of every field inside a State.
`diff(before, after)` — highlights what a single token changed.

In [ ]:
def inspect(state, label="State"):
    """Pretty-print every field in a State object."""
    print(f"\n{'=' * 65}")
    print(f"  {label}")
    print(f"{'=' * 65}")

    # Identity
    print(f"  tokens applied : {state.token_sequence}")
    print(f"  class counts   : {state.class_counts}")
    print(f"  terminated     : {state.terminated}")
    print(f"  depth          : {state.depth}")
    print(f"  n_models       : {state.n_models_applied}")
    print(f"  mase           : {state.mase}")
    print()

    # Core arrays
    print("  Core arrays:")
    for aname in ["original_history", "original_future",
                   "active_target_base", "current_target"]:
        arr = getattr(state, aname)
        print(f"    {aname:25s} {str(arr.shape):15s}  "
              f"range [{arr.min():.4f}, {arr.max():.4f}]  "
              f"mean {arr.mean():.4f}")
    print()

    # Feature stores
    def _show_store(title, d):
        if not d:
            print(f"  {title}: (empty)")
            return
        print(f"  {title}:")
        for k, v in d.items():
            if hasattr(v, "shape"):
                print(f"    {k:30s} {str(v.shape):15s}  "
                      f"range [{v.min():.4f}, {v.max():.4f}]")
            else:
                print(f"    {k:30s} {type(v).__name__}: {v}")

    _show_store("historical_features", state.historical_features)
    _show_store("future_features", state.future_features)
    _show_store("features (legacy)", state.features)
    print()

    # Metadata & flags
    if state.metadata:
        print("  metadata:")
        for k, v in state.metadata.items():
            vstr = f"array {v.shape}" if hasattr(v, "shape") else f"{v}"
            print(f"    {k}: {vstr}")
    if state.flags:
        print(f"  flags: {state.flags}")
    else:
        print("  flags: (empty)")
    print()

    # Transform stack
    if state.transform_stack:
        print(f"  Transform stack ({len(state.transform_stack)}):")
        for i, t in enumerate(state.transform_stack):
            inv = "yes" if t.inverse_fn else "no"
            print(f"    [{i}] {t.name}  (inverse: {inv}, affects: {t.affects})")
            if t.params:
                for k, v in t.params.items():
                    vstr = f"array {v.shape}" if hasattr(v, "shape") else f"{v}"
                    print(f"         {k}: {vstr}")
    else:
        print("  Transform stack: (empty)")
    print()

    # Prediction stack
    if state.prediction_stack:
        print(f"  Prediction stack ({len(state.prediction_stack)}):")
        for i, (pname, pred) in enumerate(
            zip(state.prediction_names, state.prediction_stack)
        ):
            print(f"    [{i}] {pname:20s} {str(pred.shape):15s}  "
                  f"range [{pred.min():.4f}, {pred.max():.4f}]  "
                  f"mean {pred.mean():.4f}")
    else:
        print("  Prediction stack: (empty)")
    print()

    if state.pending_conditions:
        print(f"  pending_conditions: {len(state.pending_conditions)} inverse fn(s)")
    print(f"  log entries: {len(state.log)}")
    print(f"{'=' * 65}\n")


def diff(before, after, label=""):
    """Show what changed between two State snapshots."""
    title = f"Diff: {label}" if label else "Diff"
    print(f"\n{'─' * 55}")
    print(f"  {title}")
    print(f"{'─' * 55}")

    new_tokens = after.token_sequence[len(before.token_sequence):]
    if new_tokens:
        print(f"  + tokens: {new_tokens}")

    for store_name in ("historical_features", "future_features"):
        old_keys = set(getattr(before, store_name).keys())
        new_keys = set(getattr(after, store_name).keys())
        for k in sorted(new_keys - old_keys):
            v = getattr(after, store_name)[k]
            print(f"  + {store_name}.{k}  {v.shape}")

    n_old = len(before.transform_stack)
    for t in after.transform_stack[n_old:]:
        print(f"  + transform: {t.name}  (affects: {t.affects})")

    n_old_pred = len(before.prediction_stack)
    for pname, pred in zip(
        after.prediction_names[n_old_pred:],
        after.prediction_stack[n_old_pred:],
    ):
        print(f"  + prediction: {pname}  {pred.shape}")

    if not np.array_equal(before.active_target_base, after.active_target_base):
        print(f"  ~ active_target_base changed")
    if not np.array_equal(before.current_target, after.current_target):
        resid_norm = np.linalg.norm(after.current_target)
        print(f"  ~ current_target changed  (||residual|| = {resid_norm:.4f})")

    new_flags = {k: v for k, v in after.flags.items() if k not in before.flags}
    if new_flags:
        print(f"  + flags: {new_flags}")
    print(f"{'─' * 55}\n")


print("inspect() and diff() ready.")

---
## Part 5 — Data Augmentation Token (example custom token)

A full working token that jitters + smooths the history before normalization.
Lives right here in the notebook — edit it freely.

In [ ]:
class DataAugmentationToken(TransformToken):
    """Augment the raw history with jitter and/or smoothing.

    Applies on original_history so downstream tokens
    (ZNormalization, models) see the augmented version.
    """

    name        = "DataAugmentation"
    token_class = "cleaning"
    max_uses    = 1
    reads       = ("raw_history",)
    writes      = ("raw_history", "augmented_history")
    description = "Jitter + smooth the history for robustness."

    def __init__(self, jitter_sigma=0.03, smooth_window=5, seed=42):
        self.jitter_sigma  = jitter_sigma
        self.smooth_window = smooth_window
        self.seed          = seed

    def check_specific_conditions(self, state):
        if state.flags.get("augmented", False):
            return False
        return super().check_specific_conditions(state)

    def apply(self, state):
        state = state.copy()
        rng  = np.random.default_rng(self.seed)
        hist = state.original_history.copy()   # (n_samples, T)

        # Jitter
        if self.jitter_sigma > 0:
            for i in range(hist.shape[0]):
                scale = self.jitter_sigma * np.std(hist[i])
                if scale < 1e-12:
                    scale = self.jitter_sigma
                hist[i] += rng.normal(0.0, scale, size=hist.shape[1])

        # Smooth
        if self.smooth_window > 1:
            w = self.smooth_window
            kernel = np.ones(w) / w
            for i in range(hist.shape[0]):
                pad_l = w // 2
                pad_r = w - 1 - pad_l
                padded = np.pad(hist[i], (pad_l, pad_r), mode="edge")
                hist[i] = np.convolve(padded, kernel, mode="valid")

        # Store
        state.original_history = hist
        state.features["raw_history"] = hist.copy()
        state.add_historical_feature("augmented_history", hist)

        state.register_transform(
            name=self.name,
            inverse_fn=None,
            params={
                "jitter_sigma":  self.jitter_sigma,
                "smooth_window": self.smooth_window,
                "seed":          self.seed,
            },
            affects="feature",
        )
        state.flags["augmented"] = True

        self._log_execution(
            state,
            reads={"original_history": state.original_history.shape},
            writes={"augmented_history": hist.shape},
        )
        return state


print("DataAugmentationToken defined.")

---
## Part 6 — Token Registry

All tokens in one place. Add your own here.

In [ ]:
TOKENS = {
    'DataAugmentation': DataAugmentationToken(jitter_sigma=0.03, smooth_window=5),
    'ZNormalization':   ZNormalizationToken(),
    'kernel_rbf':       KernelRBFToken(),
}

print('Registered tokens:', list(TOKENS.keys()))

---
## Part 7 — Define & Run Sequence

Edit `TOKEN_SEQUENCE`, then run. Full `inspect()` + `diff()` after every token.

In [ ]:
TOKEN_SEQUENCE = [
    'DataAugmentation',
    'ZNormalization',
    'kernel_rbf',
]

In [ ]:
state = State(H, F)
snapshots = {"init": state.copy()}
inspect(state, "INIT")

for i, name in enumerate(TOKEN_SEQUENCE):
    token = TOKENS[name]
    ok = token.can_apply(state)

    print(f"\n{'#' * 65}")
    print(f"  Step {i+1}/{len(TOKEN_SEQUENCE)}: {name}   can_apply = {ok}")
    print(f"{'#' * 65}")

    if not ok:
        print(f"  SKIPPED - token cannot apply in current state.")
        continue

    before = state.copy()
    state  = token.apply(state)
    snapshots[name] = state.copy()

    inspect(state, f"AFTER  {name}")
    diff(before, state, label=name)

print("\nAll snapshots:", list(snapshots.keys()))

---
## Part 8 — Evaluation: Forecast vs Reality

Compare the final forecast against the real future using MASE, CRPS, RMSE, and relRMSE.

In [ ]:
from utils.kernels import rmse, relative_rmse, mase

# CRPS via properscoring
try:
    from properscoring import crps_ensemble
    def crps_per_sample(y_true, y_pred):
        """CRPS for a deterministic forecast (degenerate ensemble of size 1)."""
        scores = []
        for i in range(y_true.shape[0]):
            # crps_ensemble expects (obs_scalar, ensemble_1d)
            s = np.mean([crps_ensemble(y_true[i, t], y_pred[i:i+1, t])
                         for t in range(y_true.shape[1])])
            scores.append(s)
        return np.array(scores)
    print("Using properscoring for CRPS.")
except ImportError:
    def crps_per_sample(y_true, y_pred):
        """Fallback: CRPS for deterministic forecast = MAE."""
        return np.mean(np.abs(y_true - y_pred), axis=1)
    print("properscoring not found, CRPS = MAE fallback.")


def evaluate(state, H, F, label=""):
    """Compute metrics, print table, plot forecast vs actual."""
    forecast = state.get_final_prediction()
    n = forecast.shape[0]

    rmses     = np.array([rmse(F[i], forecast[i]) for i in range(n)])
    rel_rmses = np.array([relative_rmse(F[i], forecast[i]) for i in range(n)])
    mases     = np.array([mase(F[i], forecast[i], H[i]) for i in range(n)])
    crps_vals = crps_per_sample(F, forecast)

    title = f"Evaluation: {label}" if label else "Evaluation"
    print(f"\n{'=' * 60}")
    print(f"  {title}   ({n} samples)")
    print(f"{'=' * 60}")
    print(f"  {'Metric':<18s} {'Mean':>10s} {'Median':>10s} {'Std':>10s}")
    print(f"  {'---' * 16}")
    for mname, vals in [("RMSE", rmses), ("relRMSE", rel_rmses),
                        ("MASE", mases), ("CRPS", crps_vals)]:
        print(f"  {mname:<18s} {vals.mean():10.4f} {np.median(vals):10.4f} {vals.std():10.4f}")
    print(f"{'=' * 60}\n")

    # Plot
    n_plot = min(6, n)
    fig, axes = plt.subplots(n_plot, 1, figsize=(12, 2.5 * n_plot), sharex=False)
    if n_plot == 1:
        axes = [axes]

    context = min(200, H.shape[1])
    for idx in range(n_plot):
        ax = axes[idx]
        h_ctx = H[idx, -context:]
        t_hist = np.arange(-len(h_ctx), 0)
        t_fut  = np.arange(0, F.shape[1])

        ax.plot(t_hist, h_ctx, color="steelblue", alpha=0.5, label="history")
        ax.plot(t_fut, F[idx], color="black", linewidth=1.5, label="actual")
        ax.plot(t_fut, forecast[idx], color="crimson", linewidth=1.5,
                linestyle="--", label="forecast")
        ax.axvline(0, color="gray", linestyle=":", alpha=0.5)
        ax.set_title(f"Sample {idx}  |  MASE={mases[idx]:.3f}  "
                     f"relRMSE={rel_rmses[idx]:.3f}  CRPS={crps_vals[idx]:.3f}",
                     fontsize=10)
        if idx == 0:
            ax.legend(fontsize=8, loc="upper left")

    plt.tight_layout()
    plt.show()

    return {"forecast": forecast, "rmse": rmses, "rel_rmse": rel_rmses,
            "mase": mases, "crps": crps_vals}


print("evaluate() ready.")

In [ ]:
# Run evaluation on the main data
metrics = evaluate(state, H, F, label='run data')

### Evaluate on held-out data

In [ ]:
# Run the same token sequence on holdout data
state_ho = State(H_holdout, F_holdout)
for name in TOKEN_SEQUENCE:
    token = TOKENS[name]
    if token.can_apply(state_ho):
        state_ho = token.apply(state_ho)
    else:
        print(f"WARNING: {name} cannot apply on holdout state")

metrics_ho = evaluate(state_ho, H_holdout, F_holdout, label="held-out data")

---
## Part 9 — Execution Log

In [ ]:
state.print_log()

---
## Part 10 — Re-inspect Any Snapshot

Change the key to look at state after any token.

In [ ]:
# Pick any snapshot
inspect(snapshots['ZNormalization'], 'snapshot: ZNormalization')

In [ ]:
# Compare any two snapshots
diff(snapshots['init'], snapshots['DataAugmentation'], label='init -> DataAugmentation')

---
---
## Part 11 — New Token Scratch Space

Prototype new tokens here.

**Base classes available:**
- `TransformToken` — modifies target/features (runs before models)
- `FeatureToken` — adds features to the state
- `ModelToken` — fits a model and pushes predictions
- `Token` — raw base, no constraints

Minimal template below.

In [ ]:
# Token template - uncomment and edit

# class MyNewToken(TransformToken):
#     name        = 'my_new_token'
#     token_class = 'transform'
#     max_uses    = 1
#     reads       = ('raw_history',)
#     writes      = ('my_feature',)
#     description = 'One-line description.'
#
#     def check_specific_conditions(self, state):
#         return 'my_flag' not in state.flags
#
#     def apply(self, state):
#         state = state.copy()
#         # Your logic here
#         # feat = some_computation(state.original_history)
#         # state.add_historical_feature('my_feature', feat)
#         # state.flags['my_flag'] = True
#         self._log_execution(state,
#             reads={'original_history': state.original_history.shape},
#             writes={'my_feature': 'computed'})
#         return state

In [ ]:
# Quick test on a small batch

# tok = MyNewToken()
# test_state = State(H[:4], F[:4])
# print('can_apply:', tok.can_apply(test_state))
# result = tok.apply(test_state)
# inspect(result, 'after MyNewToken')

In [ ]:
# Register and re-run

# TOKENS['my_new_token'] = MyNewToken()
# TOKEN_SEQUENCE.append('my_new_token')
# # Re-run Part 7 cells above